# Day 10 - Convolutional Neural Networks

A simple CNN on the **CIFAR-100** dataset using PyTorch.

- 3 input channels (RGB)
- 5×5 filter, stride 1
- Sigmoid activation on the fully connected layer

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## Load CIFAR-100 Dataset

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])

train_data = datasets.CIFAR100(root='./data', train=True,  download=True, transform=transform)
test_data  = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

## Define CNN Architecture

- `Conv2d(3, 16, kernel_size=5, stride=1)` — 3 input channels, 5×5 filter
- `MaxPool2d(2, 2)` — halves spatial dimensions
- `Linear(16*14*14, 256)` with Sigmoid activation
- `Linear(256, 100)` — 100 output classes

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv    = nn.Conv2d(3, 16, kernel_size=5, stride=1)
        self.pool    = nn.MaxPool2d(2, 2)
        self.fc1     = nn.Linear(16 * 14 * 14, 256)
        self.fc2     = nn.Linear(256, 100)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.pool(torch.relu(self.conv(x)))
        x = x.view(x.size(0), -1)
        x = self.sigmoid(self.fc1(x))
        x = self.fc2(x)
        return x

## Train the Model (5 Epochs)

In [ ]:
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)
    print(f'Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, Accuracy={100*correct/total:.2f}%')

## Evaluate on Test Set

In [ ]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)
print(f'Test Accuracy: {100*correct/total:.2f}%')